# 03. 날씨 데이터 처리 과정

[Colab 에서 열기](https://colab.research.google.com/github/haechan-kim0901/olpark-onestop-guide/blob/main/notebooks/03_weather_api.ipynb)

야외에서 입장을 기다리는 관객에게 **지금 기온, 체감온도, 비 소식, 앞으로 6시간 날씨**를 보여 주기 위해
기상청 자료를 받아 웹 화면용으로 바꿉니다. 이 노트북은 그 과정을 한 단계씩 나눠서 보여 줍니다.

**실행하지 않아도 읽을 수 있습니다.** 아래 표와 숫자는 미리 실행해서 저장해 둔 결과입니다 (기준 시각은 1-3단계 출력에 나옵니다).

### 전체 흐름

```
기상청 API 2개 (올림픽공원 격자 한 칸)
      │  GitHub Actions 가 15분마다 scripts/update_weather.py 실행 (매시 7, 22, 37, 52분)
      ▼
data/raw/weather/          ← 전처리 전: 받은 그대로 (한 줄 = 항목 하나)
      │  ① 세로 표를 시각별 가로 표로  ② 글자를 숫자로  ③ 체감온도 계산  ④ 코드를 말로
      ▼
docs/data/weather.json     ← 웹 화면이 읽는 파일
data/processed/weather_now.csv, weather_hours.csv  ← 팀 확인용 표
```

| 단계 | 하는 일 | 한 줄 설명 |
|---|---|---|
| 0 | 준비 | 모드 고르기 |
| 1 | 가져오기 | 위치를 격자로 바꾸고, 가장 최근 발표 시각을 골라 API 2개 받기 |
| 2 | 정리 | 시각별 표로 바꾸기, 강수량 글자를 숫자로, 체감온도, 코드 풀이 |
| 3 | 안정성 | 오류가 나도 화면이 망가지지 않게 하는 장치, 결과 점검 |
| 4 | 결과 형태 | 화면용 JSON, 확인용 CSV 모양 |

> 실제 자동 갱신은 `scripts/update_weather.py` 가 합니다. 이 노트북은 같은 계산을 풀어서 보여 주고,
> 3단계에서 **노트북 결과와 스크립트 결과가 똑같은지** 직접 확인합니다.

### 처음 보는 말 풀이

| 말 | 뜻 |
|---|---|
| 초단기실황 | 기상청이 **지금 관측한 값** (매시 정각 자료) |
| 초단기예보 | 앞으로 **6시간** 예보 (매시 30분 자료) |
| 격자 | 기상청은 전국을 가로세로 5km 칸으로 나눠 자료를 줍니다. 칸 번호가 `nx`, `ny` |
| 발표 시각 (`base_time`) | 그 자료를 만든 시각. 요청할 때 이 값을 같이 보내야 합니다 |
| 체감온도 | 바람, 습도 때문에 실제로 느끼는 온도 |
| 코드 | 기상청이 날씨를 숫자로 줌. 예: 하늘 상태 1 맑음, 3 구름많음, 4 흐림 |

## 0. 준비

- `"saved"` (기본): GitHub Actions 가 저장소에 올려 둔 **가장 최근 원본**을 씁니다. 인증키가 필요 없습니다.
- `"live"`: 지금 바로 기상청 API 를 부릅니다. 인증키가 필요합니다 (물품보관함과 같은 공공데이터포털 키). 환경 변수 `DATA_GO_KR_KEY` 에서 먼저 찾고, 없으면 입력 칸이 뜹니다.

In [1]:
MODE = "saved"       # "saved" 또는 "live"
SAVE_FILES = False   # True 로 바꾸면 4단계 결과를 output/ 폴더에 파일로 저장 (Colab 은 zip 으로 내려받기)
SCRIPT_FILE = "update_weather.py"

import json, math, os, re, sys, urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
KST = timezone(timedelta(hours=9))

# 저장소 폴더 찾기: 내 컴퓨터에서 notebooks/ 안에서 열었으면 바로 위 폴더가 저장소
ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / "scripts" / SCRIPT_FILE).exists() and (p / "docs" / "index.html").exists()), None)
GITHUB_RAW = "https://raw.githubusercontent.com/haechan-kim0901/olpark-onestop-guide/main"

def read_repo_text(rel):
    """저장소 파일 읽기. 내 컴퓨터에 저장소가 있으면 그 파일, 없으면(Colab) GitHub 에서 최신 파일을 받음."""
    if ROOT:
        return (ROOT / rel).read_text(encoding="utf-8-sig")
    with urllib.request.urlopen(f"{GITHUB_RAW}/{rel}", timeout=30) as r:
        return r.read().decode("utf-8-sig")

# 자동 갱신 스크립트를 불러옴 (3단계에서 "노트북 결과 = 스크립트 결과" 를 확인할 때 씀)
if ROOT:
    sys.path.insert(0, str(ROOT / "scripts"))
else:
    os.makedirs("repo_scripts", exist_ok=True)   # Colab: 스크립트를 GitHub 에서 받아 이 폴더에 둠
    for f in ("csv_out.py", SCRIPT_FILE):
        Path("repo_scripts", f).write_text(read_repo_text(f"scripts/{f}"), encoding="utf-8")
    sys.path.insert(0, "repo_scripts")
import importlib
script = importlib.import_module(SCRIPT_FILE[:-3])

print("파일 읽는 곳:", "내 컴퓨터의 저장소 폴더" if ROOT else "GitHub 저장소 (Colab 등)")
print("모드:", MODE)

파일 읽는 곳: 내 컴퓨터의 저장소 폴더
모드: saved


In [2]:
# live 모드일 때만 인증키를 씀. 키는 코드, 파일, 출력 어디에도 남기지 않음
KEY = None
if MODE == "live":
    from urllib.parse import unquote
    KEY = os.environ.get("DATA_GO_KR_KEY", "").strip()   # 1순위: 환경 변수
    if not KEY:                                           # 2순위: 직접 입력 (화면에 안 보임)
        from getpass import getpass
        KEY = getpass("공공데이터포털 인증키를 붙여 넣고 Enter: ").strip()
    KEY = unquote(KEY)   # Encoding 키를 넣어도 Decoding 키로 바꿔 줌
    print("키 길이:", len(KEY), "(키 내용은 보여 주지 않습니다)")
else:
    print("saved 모드: 인증키 없이 저장소에 저장된 최근 원본을 씁니다.")

saved 모드: 인증키 없이 저장소에 저장된 최근 원본을 씁니다.


## 1. 가져오기

### 1-1. 올림픽공원 위치를 기상청 격자로 바꾸기

기상청 API 는 위도, 경도 대신 **격자 번호**를 받습니다. 기상청이 공개한 변환 공식(람베르트 도법)으로 바꿉니다.
공연장 4곳이 모두 같은 칸에 들어가서, 올림픽공원 전체에 한 칸 값을 씁니다.

In [3]:
LAT, LON = 37.5207, 127.1214   # 올림픽공원 중심 (평화의광장 부근)
places = {"올림픽공원 중심": (LAT, LON), "KSPO DOME": (37.5196, 127.1273), "올림픽홀": (37.5180, 127.1250),
          "핸드볼경기장": (37.5186, 127.1300), "88잔디마당": (37.5213, 127.1196)}
grid = pd.DataFrame([{"장소": k, "위도": a, "경도": b, "nx": script.latlon_to_grid(a, b)[0], "ny": script.latlon_to_grid(a, b)[1]}
                     for k, (a, b) in places.items()])
NX, NY = script.NX, script.NY
print(f"올림픽공원 격자: nx={NX}, ny={NY}")
grid

올림픽공원 격자: nx=62, ny=126


,장소,위도,경도,nx,ny
0,올림픽공원 중심,37.5207,127.1214,62,126
1,KSPO DOME,37.5196,127.1273,62,126
2,올림픽홀,37.5180,127.1250,62,126
3,핸드볼경기장,37.5186,127.1300,62,126
4,88잔디마당,37.5213,127.1196,62,126


### 1-2. 가장 최근 발표 시각 고르기

기상청 자료는 만들어진 뒤 조금 지나야 받을 수 있습니다.

| API | 자료 시각 | 받을 수 있는 때 |
|---|---|---|
| 초단기실황 `getUltraSrtNcst` | 매시 **정각** (17:00 등) | 정각 **40분 뒤**부터 (17:40) |
| 초단기예보 `getUltraSrtFcst` | 매시 **30분** (17:30 등) | 30분 자료가 **45분 뒤**부터 (18:15) |

그래서 "지금 시각에서 기다리는 시간만큼 빼고, 그 전의 가장 가까운 자료 시각" 을 고릅니다.
아직 안 나왔으면(결과 코드 `03` NO_DATA) **한 시간 전 자료**로 다시 요청합니다 (최대 3번).
아래 표는 실행 시각에 따라 어떤 자료를 요청하는지 보여 줍니다.

In [4]:
def pick_base(now, lag_min, minute):
    """스크립트 call_latest 의 첫 요청 시각 계산과 같음"""
    t = now - timedelta(minutes=lag_min)
    base = t.replace(minute=minute, second=0, microsecond=0)
    if base > t:
        base -= timedelta(hours=1)
    return base

examples = ["2026-11-25 17:35", "2026-11-25 17:41", "2026-11-25 17:52", "2026-11-25 18:07", "2026-11-25 18:22"]
pd.DataFrame([{"실행 시각": e,
               "초단기실황 요청": pick_base(datetime.strptime(e, "%Y-%m-%d %H:%M"), 40, 0).strftime("%H:%M 자료"),
               "초단기예보 요청": pick_base(datetime.strptime(e, "%Y-%m-%d %H:%M"), 45, 30).strftime("%H:%M 자료")}
              for e in examples])

,실행 시각,초단기실황 요청,초단기예보 요청
0,2026-11-25 17:35,16:00 자료,16:30 자료
1,2026-11-25 17:41,17:00 자료,16:30 자료
2,2026-11-25 17:52,17:00 자료,16:30 자료
3,2026-11-25 18:07,17:00 자료,16:30 자료
4,2026-11-25 18:22,17:00 자료,17:30 자료


### 1-3. API 2개 받기

주소는 `https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/` 로 시작합니다.
보내는 값: `serviceKey`(인증키), `dataType=JSON`, `numOfRows=1000`, `base_date`, `base_time`(1-2단계), `nx`, `ny`(1-1단계).
정상이면 결과 코드가 `00` 입니다.

In [5]:
if MODE == "live":
    now = datetime.now(KST).replace(tzinfo=None)
    ncst_base, ncst = script.call_latest(KEY, "getUltraSrtNcst", now, 40, 0)   # 스크립트 함수 그대로 (03 이면 한 시간 전으로)
    try:
        fcst_base, fcst = script.call_latest(KEY, "getUltraSrtFcst", now, 45, 30)
    except RuntimeError as e:
        print("초단기예보 실패, 지금 날씨만 씀:", e)
        fcst_base, fcst = None, []
    raw_text = json.dumps([ncst, fcst], ensure_ascii=False)
else:
    def load(name):
        d = json.loads(read_repo_text(f"data/raw/weather/{name}.json"))
        p = d["params"]
        return datetime.strptime(p["base_date"] + p["base_time"], "%Y%m%d%H%M"), d["items"], json.dumps(d, ensure_ascii=False)
    ncst_base, ncst, t1 = load("ultra_srt_ncst")
    fcst_base, fcst, t2 = load("ultra_srt_fcst")
    raw_text = t1 + t2

print(f"초단기실황 {ncst_base:%Y-%m-%d %H:%M} 자료 {len(ncst)}줄")
print(f"초단기예보 {fcst_base:%Y-%m-%d %H:%M} 자료 {len(fcst)}줄" if fcst_base else "초단기예보 없음")

초단기실황 2026-11-25 17:00 자료 8줄
초단기예보 2026-11-25 16:30 자료 66줄


### 받은 원본 모양 보기
기상청 자료는 **한 줄 = 항목 하나** 인 세로 표입니다. 기온, 습도, 바람이 각각 다른 줄에 옵니다. 값도 모두 글자입니다.
(`category_ko` 열은 읽기 쉽게 우리가 덧붙인 뜻풀이입니다.)

In [6]:
df_ncst = pd.DataFrame(ncst)
df_ncst[["baseDate", "baseTime", "category", "category_ko", "obsrValue"]] if "category_ko" in df_ncst else df_ncst

,baseDate,baseTime,category,category_ko,obsrValue
0,20261125,1700,PTY,강수형태(코드),0
1,20261125,1700,REH,습도(%),56
2,20261125,1700,RN1,1시간 강수량(mm),0
3,20261125,1700,T1H,기온(도),7.1
4,20261125,1700,UUU,동서바람성분(m/s),0
5,20261125,1700,VEC,풍향(도),180
6,20261125,1700,VVV,남북바람성분(m/s),0.8
7,20261125,1700,WSD,풍속(m/s),0.8


In [7]:
df_fcst = pd.DataFrame(fcst)
print(f"초단기예보: 예보 시각 {df_fcst['fcstTime'].nunique()}개 x 항목 {df_fcst['category'].nunique()}개 = {len(df_fcst)}줄. 앞 8줄:")
df_fcst[["category", "fcstDate", "fcstTime", "fcstValue"]].head(8)

초단기예보: 예보 시각 6개 x 항목 11개 = 66줄. 앞 8줄:


,category,fcstDate,fcstTime,fcstValue
0,LGT,20261125,1700,0
1,LGT,20261125,1800,0
2,LGT,20261125,1900,0
3,LGT,20261125,2000,0
4,LGT,20261125,2100,0
5,LGT,20261125,2200,0
6,PTY,20261125,1700,0
7,PTY,20261125,1800,0


## 2. 정리

### 2-1. 세로 표를 시각별 가로 표로 바꾸기

화면은 "17시: 기온 7도, 흐림, 비 없음" 처럼 **시각 하나에 값 여러 개**가 필요합니다.
그래서 예보 시각을 줄로, 항목을 열로 다시 놓습니다 (엑셀의 피벗 테이블과 같은 작업). 지금 관측값도 항목별로 한 묶음으로 만듭니다.

In [8]:
obs = {i["category"]: i.get("obsrValue") for i in ncst}          # 지금 관측: 항목 → 값
by_time = {}
for i in fcst:                                                     # 예보: 시각 → {항목: 값}
    by_time.setdefault(f"{i['fcstDate']}{i['fcstTime']}", {})[i["category"]] = i.get("fcstValue")
times = sorted(by_time)[:6]                                        # 가까운 6시간만

print("지금 관측:", obs)
wide = pd.DataFrame([{"예보시각": f"{k[8:10]}:{k[10:12]}", **by_time[k]} for k in times])
wide[["예보시각", "T1H", "REH", "WSD", "SKY", "PTY", "RN1"]].rename(columns={
    "T1H": "기온 T1H", "REH": "습도 REH", "WSD": "풍속 WSD", "SKY": "하늘 SKY", "PTY": "강수형태 PTY", "RN1": "강수량 RN1"})

지금 관측: {'PTY': '0', 'REH': '56', 'RN1': '0', 'T1H': '7.1', 'UUU': '0', 'VEC': '180', 'VVV': '0.8', 'WSD': '0.8'}


,예보시각,기온 T1H,습도 REH,풍속 WSD,하늘 SKY,강수형태 PTY,강수량 RN1
0,17:00,7,55,1,4,0,강수없음
1,18:00,5,65,1,1,0,강수없음
2,19:00,5,70,2,3,0,강수없음
3,20:00,5,75,3,1,0,강수없음
4,21:00,5,75,3,1,0,강수없음
5,22:00,5,80,3,1,0,강수없음


### 2-2. 글자를 숫자로 바꾸기

기온 `"7.1"` 처럼 숫자 글자는 바로 숫자로 바꿉니다. 숫자가 아니면 빈 값(None)으로 둡니다.

예보 강수량(`RN1`)은 **"강수없음", "1mm 미만", "30.0~50.0mm"** 같은 말로 옵니다. 그래프와 안내 문구에 쓰려고 아래 규칙으로 숫자로 바꿉니다.

In [9]:
def num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def rain_mm(v):
    s = str(v or "").strip()
    if not s or s in ("강수없음", "0"):
        return 0.0
    if "미만" in s:
        return 0.5                      # 1mm 미만 → 0.5mm 로 봄
    s = s.replace("mm", "").replace("이상", "").strip()
    if "~" in s:
        s = s.split("~")[0]             # 30.0~50.0mm → 아래쪽 값 30.0
    return num(s) or 0.0

samples = ["강수없음", "1mm 미만", "1.0mm", "6.5mm", "30.0~50.0mm", "50.0mm 이상"]
pd.DataFrame([{"기상청 글자": s, "바꾼 숫자(mm)": rain_mm(s), "스크립트 결과": script.rain_mm(s)} for s in samples])

,기상청 글자,바꾼 숫자(mm),스크립트 결과
0,강수없음,0.0,0.0
1,1mm 미만,0.5,0.5
2,1.0mm,1.0,1.0
3,6.5mm,6.5,6.5
4,30.0~50.0mm,30.0,30.0
5,50.0mm 이상,50.0,50.0


### 2-3. 체감온도 계산

기상청이 쓰는 공식을 그대로 씁니다. 계절에 따라 공식이 다릅니다.

| 계절 | 공식 | 쓰는 값 | 언제 계산하나 |
|---|---|---|---|
| 겨울 쪽 (10월~4월) | 바람냉각 체감온도 | 기온, 풍속 | 기온 10도 이하이고 바람이 1.3m/s 이상일 때. 아니면 기온 그대로 |
| 여름 (5월~9월) | 여름철 체감온도 | 기온, 습도 | 항상 (습도 없으면 기온 그대로) |

In [10]:
def feels_like(t, rh, wind_ms, month):
    if t is None:
        return None
    if 5 <= month <= 9:                                    # 여름철 공식 (습구온도 tw 를 먼저 구함)
        if rh is None:
            return t
        tw = (t * math.atan(0.151977 * (rh + 8.313659) ** 0.5) + math.atan(t + rh) - math.atan(rh - 1.67633)
              + 0.00391838 * rh ** 1.5 * math.atan(0.023101 * rh) - 4.686035)
        return -0.2442 + 0.55399 * tw + 0.45535 * t - 0.0022 * tw ** 2 + 0.00278 * tw * t + 3.0
    if wind_ms is None or t > 10 or wind_ms < 1.3:         # 바람냉각이 의미 없는 조건
        return t
    v = (wind_ms * 3.6) ** 0.16                            # m/s → km/h
    return 13.12 + 0.6215 * t - 11.37 * v + 0.3965 * t * v

cases = [("11월, 바람 약함", 7.1, 56, 0.8, 11), ("12월, 바람 셈", 2.0, 50, 4.0, 12), ("1월 한파", -8.0, 40, 6.0, 1),
         ("11월, 10도 넘음", 12.0, 60, 5.0, 11), ("8월, 습함", 31.0, 80, 2.0, 8), ("6월, 건조", 28.0, 30, 2.0, 6)]
pd.DataFrame([{"예시": n, "기온": t, "습도(%)": h, "풍속(m/s)": w, "월": m,
               "체감온도": round(feels_like(t, h, w, m), 1), "스크립트 결과": round(script.feels_like(t, h, w, m), 1)}
              for n, t, h, w, m in cases])

,예시,기온,습도(%),풍속(m/s),월,체감온도,스크립트 결과
0,"11월, 바람 약함",7.1,56,0.8,11,7.1,7.1
1,"12월, 바람 셈",2.0,50,4.0,12,-1.8,-1.8
2,1월 한파,-8.0,40,6.0,1,-15.6,-15.6
3,"11월, 10도 넘음",12.0,60,5.0,11,12.0,12.0
4,"8월, 습함",31.0,80,2.0,8,33.1,33.1
5,"6월, 건조",28.0,30,2.0,6,25.5,25.5


### 2-4. 코드를 말로, 화면 형식으로 묶기

| 항목 | 코드 |
|---|---|
| 하늘 상태 `SKY` | 1 맑음, 3 구름많음, 4 흐림 |
| 강수 형태 `PTY` | 0 없음, 1 비, 2 비/눈, 3 눈, 5 빗방울, 6 빗방울눈날림, 7 눈날림 |

화면용 파일에는 코드 숫자를 그대로 넣고 (화면이 한국어, 영어로 각각 풀어 씀), 확인용 CSV 에는 말로 풀어 넣습니다.
지금 관측(실황)에는 하늘 상태가 없어서 **가장 가까운 예보 시각의 하늘 상태**를 씁니다. 소수점은 한 자리로 맞춥니다.

In [11]:
SKY_KO = {1: "맑음", 3: "구름많음", 4: "흐림"}
PTY_KO = {0: "없음", 1: "비", 2: "비/눈", 3: "눈", 5: "빗방울", 6: "빗방울눈날림", 7: "눈날림"}

def r1(v):
    return None if v is None else round(v, 1)

t, rh, ws = num(obs.get("T1H")), num(obs.get("REH")), num(obs.get("WSD"))
hours = []
for k in times:
    f = by_time[k]
    ft = num(f.get("T1H"))
    hours.append({
        "time": f"{k[8:10]}:{k[10:12]}",
        "temp": r1(ft),
        "feels": r1(feels_like(ft, num(f.get("REH")), num(f.get("WSD")), int(k[4:6]))),
        "sky": int(num(f.get("SKY")) or 1),
        "pty": int(num(f.get("PTY")) or 0),
        "rain_mm": rain_mm(f.get("RN1")),
    })

out = {
    "sample": False,
    "source": "기상청_단기예보 ((구)_동네예보) 조회서비스 초단기실황, 초단기예보 (공공데이터포털)",
    "grid": {"nx": NX, "ny": NY, "name": "서울 송파구 방이동 (올림픽공원)"},
    "fetched_at": ncst_base.strftime("%Y-%m-%d %H:%M"),
    "forecast_base": fcst_base.strftime("%Y-%m-%d %H:%M") if fcst_base else "",
    "now": {
        "temp": r1(t),
        "feels": r1(feels_like(t, rh, ws, ncst_base.month)),
        "humidity": rh,
        "wind_ms": ws,
        "rain_1h_mm": num(obs.get("RN1")) or 0.0,
        "pty": int(num(obs.get("PTY")) or 0),
        "sky": hours[0]["sky"] if hours else 1,
    },
    "hours": hours,
}
n = out["now"]
print(f"{out['fetched_at']} 관측: {n['temp']}도 (체감 {n['feels']}도), 습도 {n['humidity']}%, 바람 {n['wind_ms']}m/s, "
      f"{SKY_KO.get(n['sky'])}, 강수 {PTY_KO.get(n['pty'])}")
pd.DataFrame(hours).assign(하늘=lambda d: d["sky"].map(SKY_KO), 강수=lambda d: d["pty"].map(PTY_KO))

2026-11-25 17:00 관측: 7.1도 (체감 7.1도), 습도 56.0%, 바람 0.8m/s, 흐림, 강수 없음


,time,temp,feels,sky,pty,rain_mm,하늘,강수
0,17:00,7.0,7.0,4,0,0.0,흐림,없음
1,18:00,5.0,5.0,1,0,0.0,맑음,없음
2,19:00,5.0,3.4,3,0,0.0,구름많음,없음
3,20:00,5.0,2.5,1,0,0.0,맑음,없음
4,21:00,5.0,2.5,1,0,0.0,맑음,없음
5,22:00,5.0,2.5,1,0,0.0,맑음,없음


## 3. 안정성

자동 갱신은 사람이 지켜보지 않을 때도 15분마다 돕니다. 그래서 **문제가 생겨도 화면이 깨지지 않게** 아래 장치를 넣었습니다.

| 이런 일이 생기면 | 스크립트가 하는 일 | 화면에 보이는 것 |
|---|---|---|
| 인증키가 없음 | 바로 멈춤, 기존 `weather.json` 그대로 | 직전 값 |
| 자료가 아직 안 나옴 (`03` NO_DATA) | 한 시간 전 자료로 다시 요청 (최대 3번) | 정상 값 |
| 키 오류 등으로 JSON 대신 XML 이 옴 | 앞부분을 기록하고 멈춤, 기존 파일 그대로 | 직전 값 |
| 지금 관측(실황)이 비어 있음 | 멈춤, 기존 파일 그대로 | 직전 값 |
| 실황에 기온(`T1H`)이 없음 | 오류로 멈춤, 기존 파일 그대로 | 직전 값 |
| 예보만 실패 | 지금 날씨만 저장 | 6시간 예보 칸만 빔 |
| 숫자가 아닌 값 | 빈 값(None) 처리, 하늘은 맑음(1), 강수는 없음(0)으로 | 오류 없이 표시 |
| 자료가 전과 같음 (기상청은 1시간에 한 번 바뀜) | 커밋하지 않음 | 변화 없음 |
| 관측이 2시간 넘게 오래됨 | (화면) 5분마다 다시 읽고, 2시간 넘으면 경고 | 오래된 값이라고 알림 |
| 파일을 못 읽음 | (화면) 안에 넣어 둔 샘플 값 사용, "샘플" 표시 | 샘플 값 |
| 인증키 | 코드, 원본 JSON, CSV 어디에도 저장 안 함 (GitHub Secret 에만) | 공개 저장소에 키 없음 |

아래 셀은 지금 결과가 믿을 만한지 직접 점검합니다.

In [12]:
checks = []
def check(name, ok, detail=""):
    checks.append({"점검": name, "결과": "통과" if ok else "확인 필요", "내용": detail})

check("격자가 올림픽공원 칸 (62, 126)", (NX, NY) == (62, 126), f"nx={NX}, ny={NY}")
check("지금 기온이 있음", n["temp"] is not None, f"{n['temp']}도")
check("기온이 상식 범위 (-30~45도)", all(v is not None and -30 <= v <= 45 for v in [n["temp"]] + [h["temp"] for h in hours]))
check("습도 0~100%", n["humidity"] is None or 0 <= n["humidity"] <= 100, f"{n['humidity']}%")
check("풍속 0 이상", n["wind_ms"] is None or n["wind_ms"] >= 0, f"{n['wind_ms']}m/s")
check("예보 6시간", len(hours) == 6, f"{len(hours)}시간: " + ", ".join(h["time"] for h in hours))
check("하늘, 강수 코드가 정해진 값", all(h["sky"] in SKY_KO and h["pty"] in PTY_KO for h in hours) and n["pty"] in PTY_KO)
check("강수량 0 이상", all(h["rain_mm"] >= 0 for h in hours) and n["rain_1h_mm"] >= 0)
check("시각이 팀 표준 형식 (YYYY-MM-DD HH:MM)", bool(re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}", out["fetched_at"])), out["fetched_at"])
check("원본 파일에 인증키 없음", "serviceKey" not in raw_text and (KEY is None or KEY not in raw_text))

age_min = (datetime.now(KST).replace(tzinfo=None) - ncst_base).total_seconds() / 60
checks.append({"점검": "얼마나 최근 관측인가 (실행한 때 기준)", "결과": "통과" if age_min <= 120 else "참고",
               "내용": f"{age_min:.0f}분 전 관측" + ("" if age_min <= 120 else ", Actions 예약 실행이 늦으면 생김. 화면은 2시간 넘으면 경고")})
pd.DataFrame(checks)

,점검,결과,내용
0,"격자가 올림픽공원 칸 (62, 126)",통과,"nx=62, ny=126"
1,지금 기온이 있음,통과,7.1도
2,기온이 상식 범위 (-30~45도),통과,
3,습도 0~100%,통과,56.0%
4,풍속 0 이상,통과,0.8m/s
5,예보 6시간,통과,"6시간: 17:00, 18:00, 19:00, 20:00, 21:00, 22:00"
6,"하늘, 강수 코드가 정해진 값",통과,
7,강수량 0 이상,통과,
8,시각이 팀 표준 형식 (YYYY-MM-DD HH:MM),통과,2026-11-25 17:00
9,원본 파일에 인증키 없음,통과,


### 노트북 결과 = 자동 갱신 스크립트 결과?

같은 원본을 `scripts/update_weather.py` 의 `build()` 에 넣어서, 위에서 단계별로 만든 결과와 **한 글자도 다르지 않은지** 비교합니다.
나중에 누가 스크립트를 고쳤는데 노트북은 그대로라면, 여기서 "다름" 이 나와 바로 알 수 있습니다.

In [13]:
by_script = script.build(ncst_base, ncst, fcst_base, fcst)
by_script.pop("_설명", None)
same = by_script == out
print("스크립트 결과와 같음" if same else "스크립트 결과와 다름: 노트북을 스크립트에 맞게 고쳐야 합니다")
assert same

스크립트 결과와 같음


## 4. 결과 형태

결과는 세 가지 파일로 저장됩니다.

| 파일 | 누가 보나 | 모양 |
|---|---|---|
| `docs/data/weather.json` | 웹 화면 | `now`(지금) 한 묶음 + `hours`(6시간) 목록 |
| `data/processed/weather_now.csv` | 팀원, 기획서 | 지금 날씨 1줄 |
| `data/processed/weather_hours.csv` | 팀원, 기획서 | 6시간 예보 6줄 |

CSV 는 UTF-8 BOM 으로 저장해서 엑셀에서 한글이 깨지지 않습니다.

In [14]:
print(json.dumps({k: v for k, v in out.items() if k != "hours"} | {"hours": out["hours"][:2] + ["... (6시간)"]}, ensure_ascii=False, indent=1))

{
 "sample": false,
 "source": "기상청_단기예보 ((구)_동네예보) 조회서비스 초단기실황, 초단기예보 (공공데이터포털)",
 "grid": {
  "nx": 62,
  "ny": 126,
  "name": "서울 송파구 방이동 (올림픽공원)"
 },
 "fetched_at": "2026-11-25 17:00",
 "forecast_base": "2026-11-25 16:30",
 "now": {
  "temp": 7.1,
  "feels": 7.1,
  "humidity": 56.0,
  "wind_ms": 0.8,
  "rain_1h_mm": 0.0,
  "pty": 0,
  "sky": 4
 },
 "hours": [
  {
   "time": "17:00",
   "temp": 7.0,
   "feels": 7.0,
   "sky": 4,
   "pty": 0,
   "rain_mm": 0.0
  },
  {
   "time": "18:00",
   "temp": 5.0,
   "feels": 5.0,
   "sky": 1,
   "pty": 0,
   "rain_mm": 0.0
  },
  "... (6시간)"
 ]
}


In [15]:
# 확인용 CSV 와 같은 표 (스크립트 save_processed_csv 의 열 이름 그대로)
now_row = pd.DataFrame([{
    "관측시각": out["fetched_at"], "기온_도": n["temp"], "체감온도_도": n["feels"], "습도_퍼센트": n["humidity"],
    "풍속_ms": n["wind_ms"], "1시간강수량_mm": n["rain_1h_mm"], "강수형태": PTY_KO.get(n["pty"], n["pty"]),
    "하늘상태": SKY_KO.get(n["sky"], n["sky"]), "격자": f"nx={NX}, ny={NY}", "출처": out["source"]}])
hours_rows = pd.DataFrame([{
    "예보발표시각": out["forecast_base"], "예보시각": h["time"], "기온_도": h["temp"], "체감온도_도": h["feels"],
    "하늘상태": SKY_KO.get(h["sky"], h["sky"]), "강수형태": PTY_KO.get(h["pty"], h["pty"]), "강수량_mm": h["rain_mm"]}
    for h in hours])
print("weather_now.csv")
display(now_row.drop(columns=["출처"]))
print("weather_hours.csv")
display(hours_rows)

weather_now.csv


,관측시각,기온_도,체감온도_도,습도_퍼센트,풍속_ms,1시간강수량_mm,강수형태,하늘상태,격자
0,2026-11-25 17:00,7.1,7.1,56.0,0.8,0.0,없음,흐림,"nx=62, ny=126"


weather_hours.csv


,예보발표시각,예보시각,기온_도,체감온도_도,하늘상태,강수형태,강수량_mm
0,2026-11-25 16:30,17:00,7.0,7.0,흐림,없음,0.0
1,2026-11-25 16:30,18:00,5.0,5.0,맑음,없음,0.0
2,2026-11-25 16:30,19:00,5.0,3.4,구름많음,없음,0.0
3,2026-11-25 16:30,20:00,5.0,2.5,맑음,없음,0.0
4,2026-11-25 16:30,21:00,5.0,2.5,맑음,없음,0.0
5,2026-11-25 16:30,22:00,5.0,2.5,맑음,없음,0.0


In [16]:
# 저장소에 올라가 있는 결과 CSV 와 비교 (saved 모드일 때 같아야 정상)
if MODE == "saved":
    import io
    def same_as_repo(df, rel):
        repo = pd.read_csv(io.StringIO(read_repo_text(rel)), dtype=str).fillna("")
        mine = df.astype(str).replace({"None": ""})
        return list(repo.columns) == list(mine.columns) and repo.shape == mine.shape and (repo.values == mine.values).all()
    for df, rel in ((now_row, "data/processed/weather_now.csv"), (hours_rows, "data/processed/weather_hours.csv")):
        print(rel, "→", "저장소 파일과 같습니다." if same_as_repo(df, rel) else "다릅니다. 그사이 Actions 가 새 값을 올렸을 수 있으니 다시 실행해 보세요.")

data/processed/weather_now.csv → 저장소 파일과 같습니다.
data/processed/weather_hours.csv → 저장소 파일과 같습니다.


In [17]:
# SAVE_FILES = True 일 때만: 결과를 output/ 폴더에 저장 (Colab 이면 zip 으로 내려받기)
if SAVE_FILES:
    import shutil
    os.makedirs("output", exist_ok=True)
    with open("output/weather.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1)
    now_row.to_csv("output/weather_now.csv", index=False, encoding="utf-8-sig")
    hours_rows.to_csv("output/weather_hours.csv", index=False, encoding="utf-8-sig")
    shutil.make_archive("weather_output", "zip", "output")
    try:
        from google.colab import files
        files.download("weather_output.zip")
    except ImportError:
        print("output/ 폴더와 weather_output.zip 에 저장했습니다.")
else:
    print("파일 저장은 건너뜀 (SAVE_FILES = False). 저장소 파일은 Actions 가 15분마다 자동으로 바꿉니다.")

파일 저장은 건너뜀 (SAVE_FILES = False). 저장소 파일은 Actions 가 15분마다 자동으로 바꿉니다.


## 기획서에 쓸 수 있는 설명 (예시)

> 기상청 단기예보 조회서비스의 초단기실황, 초단기예보를 15분마다 자동 수집합니다.
> 올림픽공원 좌표를 기상청 격자(nx 62, ny 126)로 변환하고, 자료 제공 시각(실황 정시 40분 후, 예보 30분 자료 45분 후)에 맞춰 가장 최근 발표분을 요청합니다.
> 항목별 세로 자료를 시각별 표로 재구성하고, 강수량 문자 구간을 수치로 바꾸었으며, 기상청 공식으로 계절별 체감온도(겨울 바람냉각, 여름 습도 반영)를 계산했습니다.
> 수집이 실패하면 직전 값을 유지하고, 관측이 2시간 넘게 갱신되지 않으면 화면에 경고를 띄웁니다.